## Main processing pipeline
1) remove irrelevant items (non-coding/AI related)
2) remove duplicates
3) lowercase
4) language

In [1]:
import pandas as pd

# nlp libs
import string
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from langdetect import detect, LangDetectException


In [2]:
df = pd.read_csv("dataset/vibe_coding_search.csv")
print(f"Shape: {test.shape}")
print(f"Info: {test.info()}")
print(f"Null values: \n{df.isnull().sum()}")
df.head()

NameError: name 'test' is not defined

### Check for the number of unique values for each column

In [ ]:
df.nunique()

Source     6511
ID        31900
Type          2
Author    24228
Text      30550
Score       369
Date        230
dtype: int64

## Removing irrelevant content 
- list out all 6511 unique sources

In [ ]:
unique_list = df['Source'].unique().tolist()
unique_count = df['Source'].nunique()

print(f"Number of unique values: {unique_count}")
print(f"List of unique values: {unique_list}")

Number of unique values: 6511
List of unique values: ['r/u_allstacksai', 'r/lovable', 'r/SideProject', 'r/vibecoding', 'r/SaaS', 'r/Udemies', 'r/technology', 'r/PromptQuestCOM', 'r/webdev', 'r/EffortlessContent', 'r/ArtificialInteligence', 'r/cursor', 'r/NextGenAITool', 'r/ClaudeAI', 'r/microsaas', 'r/openclaw', 'r/selfhosted', 'r/rust', 'r/VibeCodingCamp', 'r/MVPLaunch', 'r/SaaSAIFounders', 'r/SaasDevelopers', 'r/shadcn', 'r/dev', 'r/u_rsrini7', 'r/udemyfreebies', 'r/neovim', 'r/startups', 'r/windsurf', 'r/webdevelopment', 'r/promptingmagic', 'r/ThinkingDeeplyAI', 'r/macbookpro', 'r/macapps', 'r/HustleGPT', 'r/AppBusiness', 'r/chrome_extensions', 'r/replit', 'r/ChatGPT', 'r/IMadeThis', 'r/AppBuilding', 'r/alphaandbetausers', 'r/OpenAI', 'r/TestMyApp', 'r/buildinpublic', 'r/ShowYourApp', 'r/Optimizely', 'r/androiddev', 'r/aipromptprogramming', 'r/AntiGravityUsers', 'r/AlgorandOfficial', 'r/LLMDevs', 'r/LocalLLaMA', 'r/EngineeringManagers', 'r/Linear', 'r/epicdotdev', 'r/MistralAI', 'r/

### Clean based on keywords (if a text does not have relevant keywords, remove that)

In [ ]:
STRONG_KEYWORDS = [
    "cursor", "copilot", "windsurf", "devin", "supermaven", "tabnine", 
    "codeium", "aider", "cline", "roo code", "bolt.new", "lovable", 
    "ghostwriter", "qodo", "deepseek", "claude 3.5", "gpt-4", "openai o1",
    "vibe coding", "prompt engineering", "agentic", "llm coding"
]

AI_TERMS = [
    "ai", "artificial intelligence", "llm", "large language model", 
    "gpt", "generative", "bot", "model", "neural", "agent", "machine learning"
]

CONTEXT_TERMS = [
    "code", "coding", "program", "programming", "software", "developer", 
    "engineer", "dev", "swe", "frontend", "backend", "full stack", 
    "web dev", "app", "application", "debug", "syntax", "ide", "repo", 
    "tech stack", "salary", "hiring", "job", "layoff", "interview", 
    "career", "junior", "senior", "tech market", "workplace"
]
def is_relevant(text):
    if not isinstance(text, str):
        return False
    
    text = text.lower()
    
    # check for strong match
    for word in STRONG_KEYWORDS:
        if word in text:
            return True
            
    # check if within any AI/context terms
    has_ai = any(term in text for term in AI_TERMS)
    has_context = any(term in text for term in CONTEXT_TERMS)
    
    if has_ai and has_context:
        return True
        
    return False

In [ ]:
df['is_relevant'] = df['Text'].apply(is_relevant)
# remove duplicates
df = df.drop_duplicates(subset=['ID'])

# Filter the DataFrame
df_clean = df[df['is_relevant'] == True].copy()

print(f"Original Records: {len(df)}")
print(f"Clean Records: {len(df_clean)}")
print(f"Dropped {len(df) - len(df_clean)} irrelevant rows.")

# Check the 'Source' column of the clean data to see if garbage subs are gone
print("\nRemaining Sources (Top 20):")
print(df_clean['Source'].value_counts().head(20))

Original Records: 31900
Clean Records: 15982
Dropped 15918 irrelevant rows.

Remaining Sources (Top 20):
Source
r/LocalLLaMA               389
r/vibecoding               380
r/ClaudeAI                 309
r/ClaudeCode               234
r/GithubCopilot            179
r/SaaS                     145
r/ArtificialInteligence    142
r/cursor                   141
r/ChatGPT                  133
r/windsurf                 130
r/developersIndia          116
r/SideProject              111
r/LocalLLM                 101
r/jobboardsearch            99
r/ExperiencedDevs           91
r/AI_Agents                 91
r/webdev                    86
r/AISEOInsider              72
r/codex                     69
r/cscareerquestions         69
Name: count, dtype: int64


In [ ]:
print(f"Info: {df_clean.info()}")

unique_list = df_clean['Source'].unique().tolist()
unique_count = df_clean['Source'].nunique()

print(f"Number of unique values: {unique_count}")
print(f"List of unique values: {unique_list}")
df_clean['Text'].head()

<class 'pandas.DataFrame'>
Index: 15982 entries, 0 to 31898
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Source       15982 non-null  str  
 1   ID           15982 non-null  str  
 2   Type         15982 non-null  str  
 3   Author       15982 non-null  str  
 4   Text         15982 non-null  str  
 5   Score        15982 non-null  int64
 6   Date         15982 non-null  str  
 7   is_relevant  15982 non-null  bool 
dtypes: bool(1), int64(1), str(6)
memory usage: 1014.5 KB
Info: None
Number of unique values: 5104
List of unique values: ['r/u_allstacksai', 'r/lovable', 'r/SideProject', 'r/vibecoding', 'r/SaaS', 'r/Udemies', 'r/technology', 'r/PromptQuestCOM', 'r/webdev', 'r/EffortlessContent', 'r/ArtificialInteligence', 'r/cursor', 'r/NextGenAITool', 'r/ClaudeAI', 'r/microsaas', 'r/openclaw', 'r/selfhosted', 'r/rust', 'r/VibeCodingCamp', 'r/MVPLaunch', 'r/SaaSAIFounders', 'r/SaasDevelopers', 'r/shadcn', 'r/dev', 'r/

0    1B GitHub commits in 2025 - why the AI coding ...
1    I vibe coded a SaaS to 615+ users in 60 days. ...
6    Library of UI components that you can copy as ...
7    Vibe coding is for engineers or rookies? To be...
8    That $1T tech wipeout is a wake-up call. What’...
Name: Text, dtype: str

## Clean text

In [ ]:
# Download necessary NLTK data (run once)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# init tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\qteam\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\qteam\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\qteam\AppData\Roaming\nltk_data...


In [ ]:
def lemmatize_text(text):
    # lowercase
    text = str(text).lower()
    # remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # remove special characters
    text = text.translate(str.maketrans('', '', string.punctuation))
    # tokenise
    tokens = text.split()
    # remove stopwords and lemamatize 
    clean_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and (len(word) > 2 or not word.isascii())
    ]
    return " ".join(clean_tokens)

def is_english(text):
    try:
        return len(text) > 10 and detect(text[:100]) == 'en'
    except:
        return False

In [ ]:
test_text = "🚀 I am CODING fast with @cursor_ai! It's better than Copilot... Check: https://vibe.dev #lazy 💀. ai"

cleaned_output = lemmatize_text(test_text)

print(f"Original: {test_text}")
print(f"Cleaned:  {cleaned_output}")

Original: 🚀 I am CODING fast with @cursor_ai! It's better than Copilot... Check: https://vibe.dev #lazy 💀. ai
Cleaned:  🚀 coding fast cursorai better copilot check lazy 💀


In [ ]:
df_clean['Clean_Text'] = df_clean['Text'].apply(lemmatize_text)
df_clean = df_clean[df_clean['Clean_Text'].str.len() > 2]
df_clean = df_clean.drop_duplicates(subset=['Clean_Text'], keep='first')
print(f"After Content Dedup: {len(df_clean)}")
output_columns = ['Source', 'ID', 'Type', 'Author', 'Text', 'Clean_Text', 'Score', 'Date']
cols_to_save = [c for c in output_columns if c in df_clean.columns]

df_clean[cols_to_save].to_csv(
    "dataset/vibe_coding_processed.csv", 
    index=False, 
    encoding='utf-8-sig', 
    escapechar='\\'
)

print(df_clean[['Text', 'Clean_Text']].head())

After Content Dedup: 14696
                                                Text  \
0  1B GitHub commits in 2025 - why the AI coding ...   
1  I vibe coded a SaaS to 615+ users in 60 days. ...   
6  Library of UI components that you can copy as ...   
7  Vibe coding is for engineers or rookies? To be...   
8  That $1T tech wipeout is a wake-up call. What’...   

                                          Clean_Text  
0  github commits 2025 coding hangover hit 2026 c...  
1  vibe coded saas 615 user day finally killed ze...  
6  library component copy prompt tired using pric...  
7  vibe coding engineer rookie begin engineer i’m...  
8  tech wipeout wakeup call what’s actual future ...  


In [ ]:
test = pd.read_csv("dataset/vibe_coding_processed.csv")
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 14696 entries, 0 to 14695
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Source      14696 non-null  str  
 1   ID          14696 non-null  str  
 2   Type        14696 non-null  str  
 3   Author      14696 non-null  str  
 4   Text        14696 non-null  str  
 5   Clean_Text  14696 non-null  str  
 6   Score       14696 non-null  int64
 7   Date        14696 non-null  str  
dtypes: int64(1), str(7)
memory usage: 918.6 KB


In [3]:
irrelevant_text = """"
[HIRING] a Tech Lead Manager! in ThoughtSpot Company: ThoughtSpot
 
 Location: India 📍
 
 Date Posted: February 06, 2026 📅
 
 Categories: #senior #engineer #PM
 
 
 
 
 Apply &amp; Description 👉 https://jobboardsearch.com/redirect?utm_source=reddit&amp;utm_medium=bot&amp;utm_id=jobboarsearch&amp;utm_term=echojobs.io&amp;rurl=aHR0cHM6Ly9lY2hvam9icy5pby9qb2IvdGhvdWdodHNwb3QtdGVjaC1sZWFkLW1hbmFnZXItN2RtbjU=
"""

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load encoder only model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Ideal query
target_query = "Discussions about vibe coding, using AI agents like Cursor or Copilot to write software."
target_embedding = model.encode(target_query)

def calculate_similarity(text):
    text_embedding = model.encode(str(text))
    similarity = util.cos_sim(target_embedding, text_embedding)
    return similarity.item()
relevant_text = "1B GitHub commits in 2025 - why the AI coding hangover hits in 2026 **The AI code generation party was fun. Now we're about to clean up the mess.**  Been thinking about this after seeing the latest GitHub stats - we hit 1 billion commits pushed in a single year, up 25% YoY. That's not sustainable velocity, that's technical debt in disguise.  Alok Nandan (GP at First Ray Ventures) just published some thoughts on this that really resonated with what we're hearing from engineering teams:  **The Day Two problem is real:** Teams poured rocket fuel into code generation (Cursor, Claude Code, Copilot) but production infrastructure hasn't kept pace. When something breaks at 2 AM, someone still needs to understand what went wrong - even if AI wrote it.  **Accountability gap:** You can't throw your hands up and say ""Claude wrote it"" when prod goes down. The engineer who merged it is still on the hook.  **Verification bottleneck:** We optimized for writing code, not reviewing it. QA and testing strategies need the same investment we gave generation tools in 2024-2025.  **The cognitive debt crisis:** Engineers shipping code in languages they don't fully understand, at velocity they can't cognitively keep up with.  **Question for the group:** Anyone else seeing this shift from generation bottlenecks to verification bottlenecks? How are you handling the gap between AI-assisted speed and actual maintainability?"
print(calculate_similarity(relevant_text))
print(calculate_similarity(irrelevant_text))
# df['relevance_score'] = df['Text'].apply(calculate_similarity)
# df_clean = df[df['relevance_score'] > 0.35] # You would tune this threshold

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1146.50it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.3284265697002411
0.17906895279884338
